In [1]:
import datetime

# Data download methods

We will review three methods of downloading data:
1. Using Pandas `read_html`
2. Using a Pandas specific library
3. Using `requests` library to access generic APIs

### Pandas provides a way to access tables on web pages via `read_html`

Visit this page and use Pandas to extract the main table: https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)

In [2]:
import pandas as pd

import requests
from io import StringIO

In [3]:
URL = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'
#tables = pd.read_html(URL)

# If the line above gives a 403 error

header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}
r = requests.get(URL, headers=header)
tables = pd.read_html(StringIO(r.text))

In [4]:
tables[2]

,Country/Territory,IMF (2025)[6],World Bank (2024)[7],United Nations (2023)[8]
0,World,117165394,111326370,100834796
1,United States,30615743,29184890,27720700
2,China[n 1],19398577,18743803,17794782
3,Germany,5013574,4659929,4525704
4,Japan,4279828,4026211,4204495
...,...,...,...,...
217,Kiribati,321,308,289
218,Marshall Islands,302,280,270
219,Nauru,172,160,176
220,Montserrat,—,—,80


### Access economics related datasets via `pandas-datareader`
Many code examples in this section are from the documentation on https://pandas-datareader.readthedocs.io/en/latest/remote_data.html

In [5]:
#!pip install pandas-datareader

In [6]:
import pandas_datareader.data as web

#### FRED (Federal Reserve Economic Data)
https://fred.stlouisfed.org/categories

Provides a large set of time series datasets: various interest rates, monetary rates, financial indicators, business surveys, etc.

In [7]:
web.DataReader('GDP', 'fred', datetime.datetime(2023, 1, 1), datetime.datetime(2026, 1, 1))

,GDP
DATE,
2023-01-01,27216.445
2023-04-01,27530.055
2023-07-01,28074.846
2023-10-01,28424.722
2024-01-01,28708.161
2024-04-01,29147.044
2024-07-01,29511.664
2024-10-01,29825.182
2025-01-01,30042.113


Navigating the data sources is not very easy, although the website for FRED does provide the ability to search for datasets easily. If you are new to the website, try this:
1. Visit https://fred.stlouisfed.org/categories for a list of all data categories
2. Pick one or more series (example: Interest Rates -> Interest Rate Spreads -> 5/10-Year Breakeven Inflation Rate (https://fred.stlouisfed.org/series/T10YIE)
3. Find the series tag name next to the English name


<div>
<img src="attachment:234e13d6-0204-4600-a40e-e506b34d3898.png" width="500"/>
</div>


Notice T10YIE as the series name. Supply the 10 year and the 5 year series names together!

In [8]:
web.DataReader(['T10YIE', 'T5YIE'], 'fred', datetime.datetime(2023, 1, 1), datetime.datetime(2024, 1, 1))

,T10YIE,T5YIE
DATE,,
2023-01-02,NaN,NaN
2023-01-03,2.26,2.29
2023-01-04,2.22,2.21
2023-01-05,2.22,2.22
2023-01-06,2.21,2.18
...,...,...
2023-12-26,2.18,2.17
2023-12-27,2.15,2.12
2023-12-28,2.16,2.11


#### Fame/French data

Fama is U Chicago's own Noble prize winning professor and French is his long time collaborator. This dataset is likely to be of limited use to anyone outside the Fin Math department.

In [9]:
from pandas_datareader.famafrench import get_available_datasets

In [10]:
get_available_datasets()[:15]

['F-F_Research_Data_Factors',
 'F-F_Research_Data_Factors_weekly',
 'F-F_Research_Data_Factors_daily',
 'F-F_Research_Data_5_Factors_2x3',
 'F-F_Research_Data_5_Factors_2x3_daily',
 'Portfolios_Formed_on_ME',
 'Portfolios_Formed_on_ME_Wout_Div',
 'Portfolios_Formed_on_ME_Daily',
 'Portfolios_Formed_on_BE-ME',
 'Portfolios_Formed_on_BE-ME_Wout_Div',
 'Portfolios_Formed_on_BE-ME_Daily',
 'Portfolios_Formed_on_OP',
 'Portfolios_Formed_on_OP_Wout_Div',
 'Portfolios_Formed_on_OP_Daily',
 'Portfolios_Formed_on_INV']

In [11]:
ds = web.DataReader('5_Industry_Portfolios', 'famafrench')
print(ds['DESCR'])

5 Industry Portfolios
---------------------

This file was created using the 202510 CRSP database. It contains value- and equal-weighted returns for 5 industry portfolios. The portfolios are constructed at the end of June. The annual returns are from January to December. Missing data are indicated by -99.99 or -999. Copyright 2025 Eugene F. Fama and Kenneth R. French

  0 : Average Value Weighted Returns -- Monthly (59 rows x 5 cols)
  1 : Average Equal Weighted Returns -- Monthly (59 rows x 5 cols)
  2 : Average Value Weighted Returns -- Annual (5 rows x 5 cols)
  3 : Average Equal Weighted Returns -- Annual (5 rows x 5 cols)
  4 : Number of Firms in Portfolios (59 rows x 5 cols)
  5 : Average Firm Size (59 rows x 5 cols)
  6 : Sum of BE / Sum of ME (6 rows x 5 cols)
  7 : Value-Weighted Average of BE/ME (6 rows x 5 cols)


/var/folders/wl/xx434fr13rs3bk0q76dx3mf00000gn/T/ipykernel_62751/3709057983.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ds = web.DataReader('5_Industry_Portfolios', 'famafrench')
/var/folders/wl/xx434fr13rs3bk0q76dx3mf00000gn/T/ipykernel_62751/3709057983.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ds = web.DataReader('5_Industry_Portfolios', 'famafrench')
/var/folders/wl/xx434fr13rs3bk0q76dx3mf00000gn/T/ipykernel_62751/3709057983.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ds = web.DataReader('5_Industry_P

In [12]:
ds[0].head()

,Cnsmr,Manuf,HiTec,Hlth,Other
Date,,,,,
2020-12,3.97,2.54,5.09,4.71,5.67
2021-01,0.68,-0.85,0.12,3.42,-2.72
2021-02,-2.04,5.95,1.65,-1.25,9.79
2021-03,5.35,7.32,0.95,-0.09,5.26
2021-04,5.81,2.48,6.07,2.96,5.70


#### World Bank data

In [13]:
from pandas_datareader import wb

In [14]:
matches = wb.search('gdp.*capita.*const')
matches

,id,name,unit,source,sourceNote,sourceOrganization,topics
691,6.0.GDPpc_constant,"GDP per capita, PPP (constant 2011 internation...",,LAC Equity Lab,GDP per capita based on purchasing power parit...,b'World Development Indicators (World Bank)',Economy & Growth
11260,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$),,World Development Indicators,Gross domestic product is the total income ear...,"b'Country official statistics, National Statis...",Economy & Growth
11262,NY.GDP.PCAP.KN,GDP per capita (constant LCU),,World Development Indicators,Gross domestic product is the total income ear...,"b'Country official statistics, National Statis...",Economy & Growth
11264,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 internation...",,World Development Indicators,This indicator provides values for gross domes...,"b'International Comparison Program (ICP), Worl...",Economy & Growth
11265,NY.GDP.PCAP.PP.KD.87,"GDP per capita, PPP (constant 1987 internation...",,WDI Database Archives,,b'',


In [15]:
wb_data = wb.download(indicator='NY.GDP.PCAP.KD', country=['US', 'CA', 'MX'], start=2005, end=2008)
wb_data

/var/folders/wl/xx434fr13rs3bk0q76dx3mf00000gn/T/ipykernel_62751/3800110738.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  wb_data = wb.download(indicator='NY.GDP.PCAP.KD', country=['US', 'CA', 'MX'], start=2005, end=2008)


NY.GDP.PCAP.KD
country       year                
Canada        2008    42067.568707
              2007    42106.872438
              2006    41663.512306
              2005    41006.222952
Mexico        2008     9826.342185
              2007     9877.695327
              2006     9813.828042
              2005     9491.517303
United States 2008    53703.962896
              2007    54152.829265
              2006    53596.315237
              2005    52649.571306

Many other data sources are avialable and described in the docs for pandas-datareader

### EDGAR data via `edgartools`

In [16]:
#!pip install edgartools

In [17]:
from edgar import Company, set_identity

In [18]:
set_identity("YOUR_ID@uchicago.edu")

In [19]:
Company("AAPL").get_financials().balance_sheet()

                                 Consolidated Balance Sheets                                 
                       As of September 27, 2025 and September 28, 2024                       
                (In millions, except shares in thousands and per share data)                 
                                                                                             
                                                                Sep 27, 2025   Sep 28, 2024  
 ─────────────────────────────────────────────────────────────────────────────────────────── 
  ASSETS:                                                                                    
    Current assets:                                                                          
        Cash and Cash Equivalents:                                   $35,934        $29,943  
          Cash                                                       $28,267        $27,199  
          Level 1 - Money market funds                      

In [20]:
Company("AAPL").get_filings(form="4")

╭──────────────────────────────────────── Filings for Apple Inc. [320193] ────────────────────────────────────────╮
│                                                                                                                 │
│                                                                              Filing                             │
│    Form        Description                                                   Date         Accession Number      │
│  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
│    4           Statement of changes in beneficial ownership                  2025-11-14   0001462356-25-0000…   │
│    4           Statement of changes in beneficial ownership                  2025-11-12   0001631982-25-0000…   │
│    4           Statement of changes in beneficial ownership                  2025-10-17   0002050912-25-0000…   │
│    4           Statement of changes in beneficial ownership           

### Accessing any API via requests (and API tokens)

In recent years, a standardized method has developed to request data from remote services via an "API" or "Application Programming Interface." This defines a standard and language/library agnostic way of requesting data (or services). Data providers will often provide "end-points," siimilar to normal web URLs.

Let's revisit the FRED website which provides lots of financial time series data. Earlier, we referenced the following URL: `https://fred.stlouisfed.org/series/T10YIE`. This URL refers to a web page that a human might visit to visually inspect data. FRED provides an API counterpart which can by used by a program to retrieve data in: `https://api.stlouisfed.org/fred/series?series_id=T10YIE&api_key=abcdefghijklmnopqrstuvwxyz123456`

Notice the `api_key=abcdefghijklmnopqrstuvwxyz123456` part; it refers to a method of authentication. We will return to this later

This API end-point is indeed just a normal URL. However, the data it returns is not a page designed for human consumption. It returns a data in a structured format.

Given a URL, you can use it various ways (not that an API key is required and will be discussed later):
1. Just paste it into the browser
2. Use a command line tool, such as `curl` to access it
3. Use a library which understands the http protocol (the same protocol used by your web browser to retrieve web pages). We will invesgitage this method further.

**Detour: API Keys**
If you use a web page, you might be asked to log in. How would you log in to a website which provides data? The whole point is that you are accessing the data programatically - what does it mean to "log in?"
API Keys are used to authenticate clients. You may have to manaully and interactively create an account on a website, create an API key and provide that API key everytime you make a programatic call. 

Here is an example of creating such a key for the FRED website

1. Create an account


<div>
<img src="attachment:29d47e51-05dc-402f-8913-01dd76bfac0d.png" width="500"/>
</div>


2. Create API keys

<div>
<img src="attachment:cc79eabc-2427-4991-8a52-c9b710f79552.png" width="200"/>
</div>


3. Continue creating an API key


<div>
<img src="attachment:86b6c503-868d-47c3-9259-9e78a5e7a5fc.png" width="200"/>
</div>


4. DO NOT SHARE THE API KEY WITH ANYONE!

<div>
<img src="attachment:3e73ee18-8cb4-4596-9343-d1bd4bc97fcc.png" width="500"/>
</div>


**KEY Security** If you share your key with anyone, they may abuse the system and you get the blame. The situation is far worse for commercial services, where a leaked key may cost of thousands of dollars (or more!). You should certainly not commit a key to version control. A common practice is to read the key from a file and to add that file to the .ignore file.

Create a file called `api_keys.txt` and format it as follows:

```json
{
"FRED":"12345myapikey67890"
}
```

In [21]:
import json

with open('./api_keys.txt') as keys_file:
    keys = json.load(keys_file)

You should be able to access your key now via the code `keys['FRED']`, without exposing the key.

**API key detour ends**

Before the detour, we had come across the following end-point: `https://api.stlouisfed.org/fred/series?series_id=T10YIE&api_key=abcdefghijklmnopqrstuvwxyz123456`

The above URL can be broken up into two parts:
- URL: `https://api.stlouisfed.org/fred/series`
- Parameters: `?series_id=T10YIE&api_key=abcdefghijklmnopqrstuvwxyz123456`

Notice that the `?` character separates the URL and the parameter list. Further, each parameter is separated by the `&` character.

One of the best, yet generic, ways of accessing this end point is via the `requests` library

In [22]:
#!pip install requests

In [23]:
import requests

In [24]:
api_url = 'https://api.stlouisfed.org/fred/series/observations'

params = {"file_type": "json", "series_id": "T10YIE", "api_key": keys['FRED']}

response = requests.get(api_url, params=params)

In [25]:
response # 200 means the API call was successful ... 404 would have been bad

<Response [200]>

In [26]:
downloaded_data = json.loads(response.text)
downloaded_data

{'realtime_start': '2025-12-04',
 'realtime_end': '2025-12-04',
 'observation_start': '1600-01-01',
 'observation_end': '9999-12-31',
 'units': 'lin',
 'output_type': 1,
 'file_type': 'json',
 'order_by': 'observation_date',
 'sort_order': 'asc',
 'count': 5981,
 'offset': 0,
 'limit': 100000,
 'observations': [{'realtime_start': '2025-12-04',
   'realtime_end': '2025-12-04',
   'date': '2003-01-02',
   'value': '1.64'},
  {'realtime_start': '2025-12-04',
   'realtime_end': '2025-12-04',
   'date': '2003-01-03',
   'value': '1.62'},
  {'realtime_start': '2025-12-04',
   'realtime_end': '2025-12-04',
   'date': '2003-01-06',
   'value': '1.63'},
  {'realtime_start': '2025-12-04',
   'realtime_end': '2025-12-04',
   'date': '2003-01-07',
   'value': '1.62'},
  {'realtime_start': '2025-12-04',
   'realtime_end': '2025-12-04',
   'date': '2003-01-08',
   'value': '1.71'},
  {'realtime_start': '2025-12-04',
   'realtime_end': '2025-12-04',
   'date': '2003-01-09',
   'value': '1.78'},
  {'r

In [27]:
import pandas as pd

In [28]:
pd.DataFrame(downloaded_data['observations'])

,realtime_start,realtime_end,date,value
0,2025-12-04,2025-12-04,2003-01-02,1.64
1,2025-12-04,2025-12-04,2003-01-03,1.62
2,2025-12-04,2025-12-04,2003-01-06,1.63
3,2025-12-04,2025-12-04,2003-01-07,1.62
4,2025-12-04,2025-12-04,2003-01-08,1.71
...,...,...,...,...
5976,2025-12-04,2025-12-04,2025-11-28,2.23
5977,2025-12-04,2025-12-04,2025-12-01,2.24
5978,2025-12-04,2025-12-04,2025-12-02,2.24
5979,2025-12-04,2025-12-04,2025-12-03,2.24
